<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c38-header.png" alt="Nextia Learning · Search and Ranking: Find the Best Match" width="100%">

# The inverted index

**[The inverted index](https://learning.nextia-ai.com/courses/search/m03/the-inverted-index/)** · Search and Ranking: Find the Best Match · Module 3, lesson 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** build an inverted index by hand, look up queries with all words or any word, and use SQLite FTS5 on the catalogue.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells (about 8 MB). No account, no key and no model download. |
| **You should know** | Ranking measures, from [Ranking measures](https://learning.nextia-ai.com/courses/search/m02/ranking-measures/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1 and matplotlib 3.11.2 (bm25s 0.3.16, PyStemmer 3.1.0, sentence-transformers 6.2.0, PyTorch 2.15.0, scikit-learn 1.9.1 and LightGBM 4.7.0 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/search/m03/the-inverted-index/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/M03-L01-the-inverted-index/the-inverted-index-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas']:
    print(name, version(name))

## Setup: get the Larkfield data

Run the next cell. It downloads the files that this lesson needs (about 9 MB) into a folder `larkfield-search`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-search` (in Colab: the **Files** panel on the left), then run the cell again.

Where the data comes from: Larkfield is an invented shop. Its products, queries and relevance judgments are derived from WANDS, the Wayfair ANnotation DataSet (MIT licence). The query log with its clicks, and the prices, stock and sale flags, are **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/data/dataset-card.md) says what is real and what is made.

In [ ]:
# Setup: download the files into larkfield-search/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/"
FILES = {
    "data/catalogue.csv.gz": "569c1c4c41403d59492e4cb22be61a7e634b052f67b4d32d12a9f14e77c08c52",
    "data/queries.csv": "51e0286150a6162502faf6502e90b89e4984949ddcb64f1e83f7e1d3251d8634",
    "compare.py": "f09c610e8e9026e0c355f693cb56cd3e178e7217c0e001a744280fb82ff867d2",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-search").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-search and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-search.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, matplotlib, and the course helper as `cp` (NDCG, recall, MRR, intervals and the comparison table). It defines a few small helpers:

- `catalogue()`, `queries()` and `judgments()` read the data: one row per product, one row per query, and `{query_id: {product_id: grade}}` with grade 2 exact, 1 partial, 0 irrelevant.
- `show(ids)` prints products with their title and class.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import compare as cp
warnings.filterwarnings("ignore")

def catalogue():
    """One row per product."""
    df = pd.read_csv("data/catalogue.csv.gz", keep_default_na=False, dtype={"title_fr": str})
    for col in ("on_sale", "in_stock"):
        df[col] = df[col].astype(str).eq("True")
    return df

def queries():
    """One row per query with judgments (train, validation and test)."""
    return pd.read_csv("data/queries.csv", keep_default_na=False)

def judgments():
    """{query_id: {product_id: grade}}: 2 exact, 1 partial, 0 irrelevant."""
    j = pd.read_csv("data/judgments.csv.gz")
    out = {}
    for q, p, g in zip(j["query_id"], j["product_id"], j["grade"]):
        out.setdefault(q, {})[p] = int(g)
    return out

def show(ids, judged=None, n=10):
    """Print products: rank, ID, title, class (and the grade when judged is given)."""
    global _CAT
    if "_CAT" not in globals():
        _CAT = catalogue().set_index("product_id")
    cat = _CAT
    names = {2: "exact", 1: "partial", 0: "irrelevant"}
    for i, p in enumerate(list(ids)[:n], 1):
        g = "" if judged is None else f"  [{names.get(judged.get(p), 'not judged')}]"
        print(f"{i:2}. {p}  {cat.loc[p, 'title'][:55]:<55}  {cat.loc[p, 'product_class'][:22]}{g}")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. An index like the one at the back of a book

A book's index lists each word with the pages where it appears. An **inverted index** does the same for products: each **token** (a word after splitting and lower case) points to its **postings**, the products that contain it. Run the next cell: an index of six products.

In [ ]:
import re
products = {
    "A": "grey outdoor cushion",
    "B": "blue outdoor cushion cover",
    "C": "garden hose with reel",
    "D": "grey sofa",
    "E": "outdoor sofa with cushions",
    "F": "hose reel cart",
}
def tokens(text):
    return [t for t in re.split(r"[^\w]+", text.lower()) if t]

index = {}
for pid, title in products.items():
    for t in set(tokens(title)):
        index.setdefault(t, set()).add(pid)
for word in sorted(index):
    print(f"{word:10} -> {sorted(index[word])}")

> **Check.** "cushion" points to A and B; "outdoor" to A, B and E. Product E has "cushions", a different token.

## 2. Looking up a query

To find products for "outdoor cushion", you do not read every product: you take the postings of each word and combine them. **All words** (AND) takes the products in every set; **any word** (OR) takes the products in at least one.

> **Predict.** Which products does "outdoor cushions" find with all words?

In [ ]:
def search_all(query):     # every word must be in the product (AND)
    sets = [index.get(t, set()) for t in tokens(query)]
    return set.intersection(*sets) if sets else set()

def search_any(query):     # any word may be in the product (OR)
    return set().union(*[index.get(t, set()) for t in tokens(query)])

for q in ["outdoor cushion", "grey cushion", "outdoor cushions"]:
    print(f"{q!r:22} all words: {sorted(search_all(q))}   any word: {sorted(search_any(q))}")

> **Check.** "outdoor cushions" finds only E with all words: "cushions" is not "cushion". Module 4 fixes this with stemming.

## 3. The same idea in SQLite

SQLite's FTS5 builds an inverted index for you. This is Larkfield's search today: every word must match, ordered by FTS5's BM25 score (the next lesson explains BM25). Run the next cell: it indexes every product and searches once.

In [ ]:
import sqlite3, time
cat = catalogue()
con = sqlite3.connect(":memory:")
con.execute("CREATE VIRTUAL TABLE products USING fts5(product_id UNINDEXED, title, description, "
            "tokenize = 'unicode61 remove_diacritics 2')")
con.executemany("INSERT INTO products VALUES (?, ?, ?)", zip(cat["product_id"], cat["title"], cat["description"]))
def fts(query, k=10, mode="and"):
    words = ['"' + w + '"' for w in tokens(query)]
    match = (" " if mode == "and" else " OR ").join(words)
    sql = "SELECT product_id FROM products WHERE products MATCH ? ORDER BY bm25(products) LIMIT ?"
    return [r[0] for r in con.execute(sql, (match, k))]
start = time.perf_counter()
hits = fts("garden hose")
print(f"{(time.perf_counter() - start) * 1000:.1f} ms for one search over {len(cat):,} products")
show(hits, n=5)

> **Check.** One search takes about a millisecond, because the index reads only the postings of "garden" and "hose".

## 4. Your turn: postings for the whole catalogue

Build `title_index`: a dictionary from each token of the product titles to the set of product IDs. Then count the products whose title has both "outdoor" and "cushion".

In [ ]:
title_index = {}
for pid, title in zip(cat["product_id"], cat["title"]):
    ...   # add pid to the posting set of every token of the title

n_both = ...   # how many products have both "outdoor" and "cushion" in their title?
print(n_both)

In [ ]:
expect('n_both', n_both, 259)

## 5. What "every word must match" costs

Run the next cell: how many English validation queries find nothing with all words, and with any word.

In [ ]:
q = queries()
english = q[(q["split"] == "validation") & (q["source"] == "wands")]["query"]
empty_and = sum(1 for text in english if not fts(text, 1, "and"))
empty_or = sum(1 for text in english if not fts(text, 1, "or"))
print(f"{len(english)} English validation queries: nothing found for {empty_and} with all words, {empty_or} with any word")

> **Check.** 24 of 60 queries find nothing with all words, 0 with any word. "Any word" never leaves the page empty here, but it finds many products: the next lesson ranks them with BM25.

In the project: `search query "garden hose" --run current` (all words) and `search query "garden hose" --run bm25`.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [BM25](https://learning.nextia-ai.com/courses/search/m03/bm25/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/search/m03/the-inverted-index/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The products, queries and judgments come from WANDS (Wayfair, MIT licence). The query log, prices, stock and sale flags are synthetic: our code made them. No real customer is in this data.